# AnNhà — CTR-GCN trên Kaggle GPU
Gắn Private Dataset chứa thư mục cache có manifest.json và các NPZ. Bật Internet để tải mã và Accelerator GPU. Kết quả ghi dưới /kaggle/working.

In [ ]:
from pathlib import Path
import subprocess
import sys
import numpy
import torch

BUNDLE = Path('/kaggle/input/annha-multisource-full-v1')
PROJECT_REF = 'research/accuracy-first-fall-detection'  # Có thể ghim SHA đã review.
SEEDS = (42, 123, 2026)
MODES = ('single', 'multi')
ROOT = Path('/kaggle/working')
PROJECT = ROOT / 'annha-source'
UPSTREAM = ROOT / 'ctr-gcn-upstream'
assert (BUNDLE / 'manifest.json').is_file(), 'Sửa BUNDLE theo đường dẫn input thật'
assert torch.cuda.is_available(), 'Chọn Accelerator GPU trước khi chạy'
print(torch.__version__, numpy.__version__, torch.cuda.get_device_name(0))

In [ ]:
def run(*args):
    subprocess.run([str(arg) for arg in args], check=True)

for folder, url, revision in (
    (PROJECT, 'https://github.com/locngodev6205/Fall-Detect-AIoT-System.git', PROJECT_REF),
    (UPSTREAM, 'https://github.com/Datyth/cross-dataset-fall-detection.git',
     'dada2669566897ac0432d681c0898bb6208938a2'),
):
    if not folder.exists():
        run('git', 'clone', url, folder)
    run('git', '-C', folder, 'fetch', 'origin', revision)
    run('git', '-C', folder, 'checkout', '--detach', 'FETCH_HEAD')
    run('git', '-C', folder, 'rev-parse', 'HEAD')

In [ ]:
outputs = ROOT / 'annha-ctr-gcn-runs'
outputs.mkdir(exist_ok=True)
for mode in MODES:
    for seed in SEEDS:
        target = outputs / f'{mode}-{seed}'
        if (target / 'evaluation.json').exists():
            continue
        resume = ['--resume'] if (target / 'last.pt').exists() else []
        run(sys.executable, PROJECT / 'scripts/ml/accuracy_multisource.py', 'train',
            BUNDLE, UPSTREAM, PROJECT / 'notebooks/ctr_gcn_kaggle_config.json', target,
            '--mode', mode, '--seed', seed, '--require-gpu', *resume)

Dùng Save Version để giữ outputs. Tải thư mục annha-ctr-gcn-runs gồm best.pt, last.pt, history.json và evaluation.json. Không công khai input hoặc checkpoint khi chưa kiểm tra quyền phân phối.